In [ ]:
"""Question 1 reproduction and independent export checks.
 
The LP matrices, objective, bounds and solve order are retained from
solve_q1.py. Only paths and exports are generalized.
 
Data convention (stated in the main text):
  The Attachment 1 timestamp of data row i marks the END of the 10-minute
  interval [(i-1)*10min, i*10min]; the 144 rows exactly tile [0:00, 24:00].
  Variable x[t] (0-based array index t) is the planned purchase of the
  interval [t*10min, (t+1)*10min].
 
Storage parameters (Appendix 1 of the problem):
  SOC bounds [1200, 10800] kWh, max charge/discharge power 5000 kW,
  bidirectional 90% efficiency (charging adds 0.9*c, discharging removes
  d/0.9), SOC(0:00) = SOC(24:00) = 6000 kWh.
"""
import argparse, csv, json, os, time
from pathlib import Path
 
import numpy as np
from scipy.optimize import linprog
from scipy.sparse import lil_matrix
from openpyxl import load_workbook
 
parser = argparse.ArgumentParser(description='Reproduce the verified Q1 LP.')
parser.add_argument('--data-dir', type=Path, default=Path('附件'))
parser.add_argument('--output-dir', type=Path, default=Path('q1_output'))
parser.add_argument('--answer', type=Path, default=None)
args = parser.parse_args()
 
BASE = args.data_dir
args.output_dir.mkdir(parents=True, exist_ok=True)
OUT = args.output_dir / 'q1_results.json'
 
N_T = 144
DT = 1 / 6
ETA = 0.9
S0 = 6000.0
S_MIN = 1200.0
S_MAX = 10800.0
P_MAX = 5000.0
E_MAX = P_MAX * DT
 
# Read source data. Attachment 1 contains powers in kW; all model energies
# are kWh. Data row i covers the interval ending at its timestamp.
wbp = load_workbook(os.path.join(BASE, '附件1.xlsx'), read_only=True, data_only=True)
wsp = wbp[wbp.sheetnames[0]]
rows = list(wsp.iter_rows(min_row=2, max_row=1 + N_T, values_only=True))
price = np.array([float(r[1]) for r in rows], dtype=float)      # yuan/kWh
load = np.array([float(r[2]) for r in rows], dtype=float)        # kW
pv = np.array([float(r[3]) for r in rows], dtype=float)          # kW
ell = load * DT                                                 # kWh per slot
g = pv * DT
 
# Variables: z = [x, c, d, w, S], each 144 long, 720 in total.
ix, ic, idd, iw, iS = 0, N_T, 2 * N_T, 3 * N_T, 4 * N_T
NVAR = 5 * N_T
 
# Objective: minimize the purchase cost at the time-of-use price.
cobj = np.zeros(NVAR)
cobj[ix:ix + N_T] = price
 
# Equalities: 144 power balances + 144 SOC recursions + 1 terminal SOC.
A = lil_matrix((2 * N_T + 1, NVAR), dtype=float)
beq = np.zeros(2 * N_T + 1)
for t in range(N_T):
    A[t, ix + t] = 1.0
    A[t, idd + t] = 1.0
    A[t, ic + t] = -1.0
    A[t, iw + t] = -1.0
    beq[t] = ell[t] - g[t]
for t in range(N_T):
    r = N_T + t
    A[r, iS + t] = 1.0
    A[r, ic + t] = -ETA
    A[r, idd + t] = 1.0 / ETA
    if t == 0:
        beq[r] = S0          # S_1 - 0.9*c_1 + d_1/0.9 = SOC(0:00)
    else:
        A[r, iS + t - 1] = -1.0
A[2 * N_T, iS + N_T - 1] = 1.0
beq[2 * N_T] = S0            # terminal SOC(24:00) = SOC(0:00)
 
bounds = ([(0.0, None)] * N_T +          # x: purchase, no upper limit
          [(0.0, E_MAX)] * N_T +         # c: charge per 10-min slot
          [(0.0, E_MAX)] * N_T +         # d: discharge per 10-min slot
          [(0.0, None)] * N_T +          # w: curtailed PV, expected 0
          [(S_MIN, S_MAX)] * N_T)         # S: end-of-slot SOC
 
t0 = time.time()
res = linprog(cobj, A_eq=A.tocsr(), b_eq=beq, bounds=bounds, method='highs')
if not res.success:
    raise RuntimeError(f'LP infeasible: {res.message}')
runtime = time.time() - t0
 
z = res.x
x = z[ix:ix + N_T].copy()
c = z[ic:ic + N_T].copy()
d = z[idd:idd + N_T].copy()
w = z[iw:iw + N_T].copy()
S = z[iS:iS + N_T].copy()
for arr in (x, c, d, w, S):
    arr[np.abs(arr) < 1e-9] = 0
 
plan_cost = float(price @ x)
 
# Reference case without storage: buy the net load at each interval price.
x_base = np.maximum(ell - g, 0.0)
baseline_cost = float(price @ x_base)
 
validation = {
    'date': '2025-01-01 (single typical day of Attachment 1)',
    'plan_slots_total': N_T, 'expected_plan_slots_total': 144,
    'all_energy_unit': 'kWh', 'dt_hours': DT,
    'eta_c': ETA, 'eta_d': ETA, 'Smin': S_MIN, 'Smax': S_MAX,
    'Pmax_kW': P_MAX, 'Emax_kWh_per_10min': E_MAX,
    'soc_start_kWh': S0, 'soc_end_kWh': float(S[-1]),
    'soc_boundary_rule': 'terminal fixed: SOC(24:00) = SOC(0:00) = 6000',
    'max_charge_kWh': float(c.max()), 'max_discharge_kWh': float(d.max()),
    'soc_min_kWh': float(S.min()), 'soc_max_kWh': float(S.max()),
    'max_balance_residual_kWh': float(np.abs(x + g + d - ell - c - w).max()),
    'max_soc_residual_kWh': None,  # filled by the independent audit below
    'total_purchase_kWh': float(x.sum()),
    'plan_cost_yuan': plan_cost,
    'total_charge_kWh': float(c.sum()), 'total_discharge_kWh': float(d.sum()),
    'total_curtailment_kWh': float(w.sum()),
    'baseline_no_storage_cost_yuan': baseline_cost,
    'savings_vs_baseline_yuan': baseline_cost - plan_cost,
    'savings_ratio_vs_baseline': (baseline_cost - plan_cost) / baseline_cost,
    'runtime_seconds': runtime,
}
 
with open(OUT, 'w', encoding='utf-8') as f:
    json.dump({'price': price.tolist(), 'load': load.tolist(), 'pv': pv.tolist(),
               'x': x.tolist(), 'c': c.tolist(), 'd': d.tolist(),
               'w': w.tolist(), 'S': S.tolist(), 'validation': validation},
              f, ensure_ascii=False)
print(json.dumps(validation, ensure_ascii=False, indent=2))
 
# Independent checks use exported trajectories, not solver status alone.
audit = {'soc_recursion': 0.0, 'power_balance': 0.0, 'terminal_soc': 0.0,
         'cost': 0.0, 'simultaneous': 0.0}
boundaries = np.r_[S0, S]
soc_residual = np.diff(boundaries) - ETA * c + d / ETA
balance = x + g + d - ell - c - w
audit['soc_recursion'] = float(np.abs(soc_residual).max())
audit['power_balance'] = float(np.abs(balance).max())
audit['terminal_soc'] = float(abs(S[-1] - S0))
audit['cost'] = abs(float(price @ x) - plan_cost)
audit['simultaneous'] = float(np.minimum(c, d).max())
validation['max_soc_residual_kWh'] = audit['soc_recursion']
 
assert min(boundaries) >= S_MIN - 1e-6
assert max(boundaries) <= S_MAX + 1e-6
assert min(c) >= -1e-6 and min(d) >= -1e-6
assert max(c) <= E_MAX + 1e-6 and max(d) <= E_MAX + 1e-6
assert min(x) >= -1e-9
assert all(v < 1e-6 for v in audit.values())
 
def tm(k):
    if k == 144:
        return '0:00+1'
    return f'{(k * 10) // 60}:{(k * 10) % 60:02d}'
 
def interval_label(start, end):
    return tm(start) + '-' + tm(end)
 
def write_csv(name, header, rows):
    with (args.output_dir / name).open('w', encoding='utf-8-sig', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(header)
        writer.writerows(rows)
 
write_csv('q1_10min.csv',
          ['period', 'x', 'c', 'd', 'soc_before', 'soc_after', 'w',
           'load', 'pv', 'price', 'cost'],
          ([interval_label(t, t + 1), x[t], c[t], d[t],
            boundaries[t], boundaries[t + 1], w[t],
            load[t], pv[t], price[t], price[t] * x[t]]
           for t in range(N_T)))
 
blocks = [f'{b * 4:02d}:00-{b * 4 + 4:02d}:00' for b in range(6)]
write_csv('q1_blocks.csv', ['block', 'charge_kWh', 'discharge_kWh'],
          ([blocks[b], float(c[b * 24:(b + 1) * 24].sum()),
            float(d[b * 24:(b + 1) * 24].sum())] for b in range(6)))
 
# The official answer template labels each row by the data-row timestamp:
# row k ('k*10-(k*10+10)') holds x[k-1], i.e. the purchase of the interval
# ENDING at that label. Read only: the supplied answer file is never changed.
def tmpl_label(k):
    def fmt(m):
        h, mm = divmod(m, 60)
        if h >= 24:
            return f'{h - 24}:{mm:02d}+1'
        return f'{h}:{mm:02d}'
    return f'{fmt(k * 10)}-{fmt(k * 10 + 10)}'
 
if args.answer is not None:
    answer = load_workbook(args.answer, read_only=True, data_only=True)
    rows = list(answer['计划购电量'].values)
    assert len(rows) == N_T + 1
    assert rows[1][0] == '0:10-0:20' and rows[N_T][0] == '0:00+1-0:10+1'
    plan_error = 0.0
    for k in range(1, N_T + 1):
        label, value = rows[k][0], rows[k][1]
        assert label == tmpl_label(k), (k, label)
        plan_error = max(plan_error, abs(float(value) - x[k - 1]))
    rows2 = list(answer['充放电量'].values)
    storage_error = 0.0
    for b in range(6):
        expected_c = float(c[b * 24:(b + 1) * 24].sum())
        expected_d = float(d[b * 24:(b + 1) * 24].sum())
        storage_error = max(storage_error, abs(float(rows2[b + 1][1]) - expected_c),
                            abs(float(rows2[b + 1][2]) - expected_d))
    storage_error = max(storage_error, abs(float(rows2[1][4]) - S0),
                        abs(float(rows2[2][4]) - float(S[-1])))
    answer.close()
    assert max(plan_error, storage_error) < 1e-3   # template stores 4 decimals
    audit.update(workbook_plan=plan_error, workbook_storage=storage_error)
 
with (args.output_dir / 'q1_checks.json').open('w', encoding='utf-8') as f:
    json.dump(audit, f, ensure_ascii=False, indent=2)
 
print('Independent checks:', json.dumps(audit, ensure_ascii=False))


In [ ]:
"""Question 2 reproduction and independent export checks.
 
The LP matrices, objective, bounds, history window and solve order are
retained from run_q2_continuous.py. Only paths and exports are generalized.
"""
import sys, os, json, time
import numpy as np
from scipy.optimize import linprog, milp, LinearConstraint, Bounds
from scipy.sparse import lil_matrix, hstack
from openpyxl import load_workbook
from datetime import datetime
 
from pathlib import Path
import argparse, csv
 
parser = argparse.ArgumentParser(description='Reproduce the verified Q2 LP.')
parser.add_argument('--data-dir', type=Path, default=Path('附件'))
parser.add_argument('--output-dir', type=Path, default=Path('q2_output'))
parser.add_argument('--answer', type=Path, default=None)
args = parser.parse_args()
BASE = args.data_dir
args.output_dir.mkdir(parents=True, exist_ok=True)
OUT = args.output_dir / 'q2_continuous_results.json' 
N_T = 144
N_S = 7
DT = 1 / 6
ETA = 0.9
S0 = 6000.0
S_MIN = 1200.0
S_MAX = 10800.0
P_MAX = 5000.0
E_MAX = P_MAX * DT
 
# Read source data. Attachment 2 contains powers in kW; all model energies are kWh.
wbp = load_workbook(os.path.join(BASE, '附件1.xlsx'), read_only=True, data_only=True)
wsp = wbp[wbp.sheetnames[0]]
price = np.array([float(wsp.cell(r, 2).value) for r in range(2, N_T + 2)], dtype=float)
bootstrap_load = np.array([float(wsp.cell(r, 3).value) for r in range(2, N_T + 2)], dtype=float) * DT
bootstrap_pv = np.array([float(wsp.cell(r, 4).value) for r in range(2, N_T + 2)], dtype=float) * DT
 
wb = load_workbook(os.path.join(BASE, '附件2.xlsx'), read_only=True, data_only=True)
def read_sheet(name):
    ws = wb[name]
    ds, arr = [], []
    for row in ws.iter_rows(min_row=2, max_row=ws.max_row, min_col=1, max_col=N_T + 1, values_only=True):
        ds.append(row[0].date() if isinstance(row[0], datetime) else row[0])
        arr.append([float(v or 0) * DT for v in row[1:]])
    return ds, np.asarray(arr, dtype=float)
 
dates, load = read_sheet('小区负载')
dates2, pv = read_sheet('光伏发电实际功率')
assert dates == dates2 and len(dates) == 365 and load.shape == (365, N_T) and pv.shape == (365, N_T)
 
# Variables: x, c, r, S, uhat, then e_s and u_s for seven scenarios.
ix = lambda k, t: k * N_T + t
BASE_E = 5 * N_T
BASE_U = BASE_E + N_S * N_T
NVAR = BASE_U + N_S * N_T
NEQ = N_T + N_S * N_T + N_T
A = lil_matrix((NEQ, NVAR), dtype=float)
row = 0
for t in range(N_T):
    A[row, ix(0, t)] = 1
    A[row, ix(2, t)] = 1
    A[row, ix(1, t)] = -1
    A[row, ix(4, t)] = -1
    row += 1
for s in range(N_S):
    for t in range(N_T):
        A[row, ix(0, t)] = 1
        A[row, ix(2, t)] = 1
        A[row, BASE_E + s * N_T + t] = 1
        A[row, ix(1, t)] = -1
        A[row, BASE_U + s * N_T + t] = -1
        row += 1
for t in range(N_T):
    A[row, ix(3, t)] = 1
    A[row, ix(1, t)] = -ETA
    A[row, ix(2, t)] = 1 / ETA
    if t > 0:
        A[row, ix(3, t - 1)] = -1
    row += 1
A = A.tocsr()
 
cobj = np.zeros(NVAR)
cobj[:N_T] = price
for s in range(N_S):
    cobj[BASE_E + s * N_T:BASE_E + (s + 1) * N_T] = 5 * price / N_S
lb = np.zeros(NVAR)
ub0 = np.full(NVAR, np.inf)
for t in range(N_T):
    lb[ix(3, t)] = S_MIN
    ub0[ix(3, t)] = S_MAX
    ub0[ix(1, t)] = E_MAX
    ub0[ix(2, t)] = E_MAX
 
def history_for_day(di):
    """Return the seven forecast/scenario profiles available before day di.
 
    For Jan 1-7 only, missing pre-year history is filled with the Attachment 1
    one-day forecast profile. From Jan 8 onward, and for every reported day,
    this is exactly the preceding seven actual days.
    """
    if di >= N_S:
        return load[di - N_S:di], pv[di - N_S:di], [dates[i].isoformat() for i in range(di - N_S, di)]
    n_actual = di
    n_boot = N_S - n_actual
    hl = np.vstack([np.tile(bootstrap_load, (n_boot, 1)), load[:n_actual]])
    hg = np.vstack([np.tile(bootstrap_pv, (n_boot, 1)), pv[:n_actual]])
    names = ['附件1预报预热'] * n_boot + [dates[i].isoformat() for i in range(n_actual)]
    return hl, hg, names
 
def solve_day(di, s_init):
    hist_l, hist_g, scenario_sources = history_for_day(di)
    lhat = hist_l.mean(axis=0)
    ghat = hist_g.mean(axis=0)
    beq = np.zeros(NEQ)
    beq[:N_T] = lhat - ghat
    beq[N_T:N_T + N_S * N_T] = (hist_l - hist_g).ravel()
    beq[N_T + N_S * N_T] = s_init
    ub = ub0.copy()
    ub[4 * N_T:5 * N_T] = ghat
    # u_s is total unutilized surplus, not pure PV curtailment; it has no g_s upper bound.
    ub[BASE_U:] = 1e9
    res = linprog(cobj, A_eq=A, b_eq=beq, bounds=np.column_stack([lb, ub]), method='highs')
    if not res.success:
        raise RuntimeError(f'LP infeasible on {dates[di]}: {res.message}')
    z = res.x
    x, c, r, S, uhat = z[:N_T], z[N_T:2 * N_T], z[2 * N_T:3 * N_T], z[3 * N_T:4 * N_T], z[4 * N_T:5 * N_T]
    e = z[BASE_E:BASE_U].reshape(N_S, N_T)
    u = z[BASE_U:].reshape(N_S, N_T)
    lact, gact = load[di], pv[di]
    eact = np.maximum(lact + c - x - gact - r, 0.0)
    uact = np.maximum(gact - (lact + c - x - r), 0.0)
    predres = x + ghat + r - lhat - c - uhat
    scenres = np.array([x + hist_g[s] + r + e[s] - hist_l[s] - c - u[s] for s in range(N_S)])
    socres = np.empty(N_T)
    socres[0] = S[0] - (s_init + ETA * c[0] - r[0] / ETA)
    socres[1:] = S[1:] - (S[:-1] + ETA * c[1:] - r[1:] / ETA)
    for arr in (x, c, r, S, uhat, e, u, eact, uact):
        arr[np.abs(arr) < 1e-9] = 0
    return {
        'date': dates[di].isoformat(), 'soc_start': float(s_init), 'soc_end': float(S[-1]),
        'x': x.tolist(), 'c': c.tolist(), 'r': r.tolist(), 'S': S.tolist(), 'uhat': uhat.tolist(),
        'forecast_load': lhat.tolist(), 'forecast_pv': ghat.tolist(),
        'actual_load': lact.tolist(), 'actual_pv': gact.tolist(),
        'e_s': e.tolist(), 'u_s': u.tolist(), 'eact': eact.tolist(), 'uact': uact.tolist(),
        'plan_cost': float(np.dot(price, x)), 'emergency_cost': float(np.dot(5 * price, eact)),
        'total_cost': float(np.dot(price, x) + np.dot(5 * price, eact)),
        'scenario_sources': scenario_sources,
        'residual_pred_max': float(np.max(np.abs(predres))), 'residual_scenario_max': float(np.max(np.abs(scenres))),
        'residual_soc_max': float(np.max(np.abs(socres))), 'max_simultaneous': float(np.max(np.minimum(c, r))),
        'all_energy_unit': 'kWh', 'dt': DT
    }
 
def interval_label(start, end):
    def tm(k):
        if k == 144:
            return '0:00+1'
        return f'{(k * 10) // 60}:{(k * 10) % 60:02d}'
    return tm(start) + '-' + tm(end)
 
def urgent_intervals(e):
    out, t = [], 0
    while t < N_T:
        if e[t] <= 1e-7:
            t += 1
            continue
        st, amount = t, 0.0
        while t < N_T and e[t] > 1e-7:
            amount += e[t]
            t += 1
        out.append({'period': interval_label(st, t), 'amount': float(amount), 'start': st, 'end': t})
    return out
 
def milp_review(rec, di):
    if rec['max_simultaneous'] <= 1e-6:
        return True
    hist_l, hist_g, _ = history_for_day(di)
    lhat, ghat = hist_l.mean(0), hist_g.mean(0)
    beq = np.zeros(NEQ)
    beq[:N_T] = lhat - ghat
    beq[N_T:N_T + N_S * N_T] = (hist_l - hist_g).ravel()
    beq[N_T + N_S * N_T] = rec['soc_start']
    Aext = hstack([A, lil_matrix((NEQ, N_T))]).tocsr()
    I = lil_matrix((2 * N_T, NVAR + N_T))
    bi = np.zeros(2 * N_T)
    for t in range(N_T):
        I[2 * t, ix(1, t)] = 1
        I[2 * t, NVAR + t] = -E_MAX
        I[2 * t + 1, ix(2, t)] = 1
        I[2 * t + 1, NVAR + t] = E_MAX
        bi[2 * t + 1] = E_MAX
    ub = ub0.copy(); ub[4 * N_T:5 * N_T] = ghat; ub[BASE_U:] = 1e9
    m = milp(np.r_[cobj, np.zeros(N_T)], integrality=np.r_[np.zeros(NVAR), np.ones(N_T)],
             bounds=Bounds(np.r_[lb, np.zeros(N_T)], np.r_[ub, np.ones(N_T)]),
             constraints=[LinearConstraint(Aext, beq, beq), LinearConstraint(I.tocsr(), -np.inf, bi)],
             options={'time_limit': 60})
    return bool(m.success)
 
results_all, results, milp_dates = [], [], []
s_init = S0
t0 = time.time()
for di in range(365):
    rec = solve_day(di, s_init)
    rec['urgent_intervals'] = urgent_intervals(np.asarray(rec['eact']))
    results_all.append(rec)
    if di >= 31:
        results.append(rec)
    if rec['max_simultaneous'] > 1e-6:
        if not milp_review(rec, di):
            raise RuntimeError(f'MILP review failed on {rec["date"]}')
        milp_dates.append(rec['date'])
    s_init = rec['soc_end']
 
# Independent validation and season/month diagnostics for the reported 334 days.
all_emergency_days = sum(bool(r['urgent_intervals']) for r in results)
all_emergency_kwh = sum(sum(r['eact']) for r in results)
all_plan_cost = sum(r['plan_cost'] for r in results)
all_emergency_cost = sum(r['emergency_cost'] for r in results)
validation = {
    'date_count': len(results), 'expected_date_count': 334, 'plan_slots_total': len(results) * N_T,
    'expected_plan_slots_total': 334 * N_T, 'warmup_days': 31, 'warmup_initial_soc': S0,
    'soc_boundary': '连续跨日：S(day+1,0)=S(day,144)', 'reported_start_soc': results[0]['soc_start'],
    'reported_end_soc': results[-1]['soc_end'], 'all_energy_unit': 'kWh', 'dt_hours': DT,
    'eta_c': ETA, 'eta_d': ETA, 'Smin': S_MIN, 'Smax': S_MAX, 'Pmax_kW': P_MAX, 'Emax_kWh_per_10min': E_MAX,
    'max_charge_kWh': max(max(r['c']) for r in results), 'max_discharge_kWh': max(max(r['r']) for r in results),
    'soc_min_kWh': min(min(r['S']) for r in results), 'soc_max_kWh': max(max(r['S']) for r in results),
    'max_soc_residual_kWh': max(r['residual_soc_max'] for r in results),
    'max_plan_balance_residual_kWh': max(r['residual_pred_max'] for r in results),
    'max_scenario_balance_residual_kWh': max(r['residual_scenario_max'] for r in results),
    'min_emergency_kWh': min(min(r['eact']) for r in results),
    'max_emergency_cost_residual': max(abs(r['emergency_cost'] - sum(5 * price[t] * r['eact'][t] for t in range(N_T))) for r in results),
    'time_mapping_ok': True, 'same_day_data_used_in_reported_forecast': False,
    'forecast_rule': '同一10分钟时段取前7天实际数据平均',
    'forecast_probability': '7个历史日等概率，每个为1/7',
    'forecast_window_first_reported_day': '2025-01-25...2025-01-31',
    'forecast_window_last_reported_day': '2025-12-24...2025-12-30',
    'warmup_rule': '2025-01-01至01-07的缺失年初历史用附件1的一日预测曲线预热；随着日期推进逐步加入已发生的实际日。2025-01-08起使用此前7个实际日。',
    'surplus_variable': 'u_s,t表示情景下总剩余未利用电量，仅约束u_s,t≥0，不设置u_s,t≤g_s,t上界',
    'emergency_days': all_emergency_days, 'emergency_day_ratio': all_emergency_days / len(results),
    'emergency_energy_kWh': all_emergency_kwh, 'plan_cost_total': all_plan_cost,
    'emergency_cost_total': all_emergency_cost, 'total_cost_total': all_plan_cost + all_emergency_cost,
    'simultaneous_charge_discharge_dates': milp_dates, 'milp_review_count': len(milp_dates),
    'runtime_seconds': time.time() - t0,
}
 
for r in results:
    r['monthly'] = r['date'][:7]
with open(OUT, 'w', encoding='utf-8') as f:
    json.dump({'price': price.tolist(), 'results': results, 'all_results': results_all, 'validation': validation}, f, ensure_ascii=False)
print(json.dumps(validation, ensure_ascii=False, indent=2))
 
# Independent checks use exported trajectories, not solver status alone.
audit = {'state': 0.0, 'actual_balance': 0.0, 'cost': 0.0,
         'cross_day': 0.0, 'simultaneous': 0.0}
previous_end = S0
for di, rec in enumerate(results_all):
    x, c, r, state = [np.asarray(rec[k]) for k in ['x', 'c', 'r', 'S']]
    e, u = [np.asarray(rec[k]) for k in ['eact', 'uact']]
    boundaries = np.r_[rec['soc_start'], state]
    residual = np.diff(boundaries) - ETA * c + r / ETA
    balance = x + pv[di] + r + e - load[di] - c - u
    cost = float(price @ x + (5 * price) @ e)
    audit['state'] = max(audit['state'], float(abs(residual).max()))
    audit['actual_balance'] = max(audit['actual_balance'],
                                  float(abs(balance).max()))
    audit['cost'] = max(audit['cost'], abs(cost - rec['total_cost']))
    audit['cross_day'] = max(audit['cross_day'],
                            abs(rec['soc_start'] - previous_end))
    audit['simultaneous'] = max(audit['simultaneous'],
                               float(np.minimum(c, r).max()))
    assert min(boundaries) >= S_MIN - 1e-6
    assert max(boundaries) <= S_MAX + 1e-6
    assert min(c) >= -1e-6 and min(r) >= -1e-6
    assert max(c) <= E_MAX + 1e-6 and max(r) <= E_MAX + 1e-6
    if di >= 7:
        expected = [dates[j].isoformat() for j in range(di-7, di)]
        assert rec['scenario_sources'] == expected
        assert all(s < rec['date'] for s in expected)
    previous_end = rec['soc_end']
assert all(v < 1e-6 for v in audit.values())
 
def write_csv(name, header, rows):
    with (args.output_dir / name).open('w', encoding='utf-8-sig',
                                       newline='') as f:
        writer = csv.writer(f)
        writer.writerow(header)
        writer.writerows(rows)
 
write_csv('q2_10min.csv',
    ['date', 'period', 'x', 'c', 'r', 'soc_before', 'soc_after',
     'load_forecast', 'pv_forecast', 'load_actual', 'pv_actual',
     'emergency', 'unused', 'price', 'plan_cost', 'emergency_cost'],
    ([rec['date'], interval_label(t, t+1), rec['x'][t], rec['c'][t],
      rec['r'][t], rec['soc_start'] if t == 0 else rec['S'][t-1],
      rec['S'][t], rec['forecast_load'][t], rec['forecast_pv'][t],
      rec['actual_load'][t], rec['actual_pv'][t], rec['eact'][t],
      rec['uact'][t], price[t], price[t] * rec['x'][t],
      5 * price[t] * rec['eact'][t]]
     for rec in results for t in range(N_T)))
write_csv('q2_daily.csv',
    ['date', 'soc_start', 'soc_end', 'plan_kWh', 'emergency_kWh',
     'plan_cost', 'emergency_cost', 'total_cost'],
    ([rec['date'], rec['soc_start'], rec['soc_end'], sum(rec['x']),
      sum(rec['eact']), rec['plan_cost'], rec['emergency_cost'],
      rec['total_cost']] for rec in results))
write_csv('q2_emergency.csv', ['date', 'period', 'kWh'],
    ([rec['date'], q['period'], q['amount']]
     for rec in results for q in rec['urgent_intervals']))
 
# The official answer template rotates the first slot to column EO.
# Read only: the supplied answer workbook is never changed.
if args.answer is not None:
    answer = load_workbook(args.answer, read_only=True, data_only=True)
    rows = list(answer['计划购电量'].values)
    assert len(rows) == len(results) + 1
    plan_error = storage_error = emergency_error = 0.0
    for row, rec in zip(rows[1:], results):
        assert row[0].date().isoformat() == rec['date']
        restored = np.asarray([row[144], *row[1:144]], dtype=float)
        plan_error = max(plan_error, float(abs(restored-rec['x']).max()))
        assert abs(row[145] - sum(rec['x'])) < 1e-6
        assert abs(row[146] - rec['plan_cost']) < 1e-6
    rows = list(answer['充放电量'].values)
    assert len(rows) == 1 + len(results) * 6
    for j, rec in enumerate(results):
        block = rows[1+j*6:7+j*6]
        assert block[0][0].date().isoformat() == rec['date']
        for k, row in enumerate(block):
            for col, key in [(2, 'c'), (3, 'r')]:
                expected = sum(rec[key][k*24:(k+1)*24])
                storage_error = max(storage_error, abs(row[col]-expected))
        assert abs(block[0][5] - rec['soc_start']) < 1e-6
        assert abs(block[1][5] - rec['soc_end']) < 1e-6
    rows = list(answer['紧急购电量'].values)[1:]
    expected = [(rec['date'], q['period'], q['amount'])
                for rec in results for q in rec['urgent_intervals']]
    assert len(rows) == len(expected)
    for row, item in zip(rows, expected):
        assert (row[0].date().isoformat(), row[1]) == item[:2]
        emergency_error = max(emergency_error, abs(row[2]-item[2]))
    answer.close()
    assert max(plan_error, storage_error, emergency_error) < 1e-6
    audit.update(workbook_plan=plan_error, workbook_storage=storage_error,
                 workbook_emergency=emergency_error)
with (args.output_dir / 'q2_checks.json').open('w', encoding='utf-8') as f:
    json.dump(audit, f, ensure_ascii=False, indent=2)
print('Independent checks:', json.dumps(audit, ensure_ascii=False))


In [ ]:
"""Question 3 reproduction and independent export checks.
 
The rolling framework, LP matrices, piecewise settlement linearization,
storage feedback executor and day-by-day solve order are retained from
solve_q3.py (main scenario B61218, terminal mode fix6000). Only paths and
exports are generalized.
 
Data conventions (stated in the main text):
  Attachment 1/2/3 data-row timestamp t marks the END of the 10-minute
  interval [t-10min, t]; the 144 rows of a day tile [0:00, 24:00].
  In Attachment 3, the column "forecast at +k hours" issued at T holds the
  PV power point value at the hour T+k (verified by lag-correlation
  analysis). Forecasts are linearly interpolated onto the 10-minute grid,
  anchored at the measured PV power at the issue time.
 
Rolling structure per day:
  00:00 planning LP over all 144 slots at the 0:00 forecast;
  06:00/12:00/18:00 adjustment LPs over the remaining slots at the fresh
  forecast, with the piecewise settlement rule
      min(x, y) at the normal price, (x - y)+ at 50%, (y - x)+ at 150%,
  linearized by the dp/dm variables;
  real-time execution fixes the planned charge/discharge actions and the
  storage feeds back: reduce charge/discharge first, then increase
  discharge/charge, then emergency purchase at 5x price, surplus is curtailed.
  Terminal SOC target 6000 kWh each day; the next day starts from the
  actual end-of-day SOC.
"""
import argparse, csv, json, os, time
from datetime import datetime
from pathlib import Path
 
import numpy as np
from scipy.optimize import linprog
from scipy.sparse import coo_matrix
from openpyxl import load_workbook
 
parser = argparse.ArgumentParser(description='Reproduce the verified Q3 strategy.')
parser.add_argument('--data-dir', type=Path, default=Path('附件'))
parser.add_argument('--output-dir', type=Path, default=Path('q3_output'))
parser.add_argument('--answer', type=Path, default=None)
args = parser.parse_args()
 
BASE = args.data_dir
args.output_dir.mkdir(parents=True, exist_ok=True)
OUT = args.output_dir / 'q3_results.json'
 
N_T = 144
DT = 1 / 6
ETA = 0.9
S0 = 6000.0
S_MIN = 1200.0
S_MAX = 10800.0
P_MAX = 5000.0
E_MAX = P_MAX * DT
K_EMG = 5.0        # emergency purchase price multiple
K_PEN = 0.5         # settlement: |y - x| pays an extra 50% on the deviation
SOC_TGT = 6000.0    # terminal mode fix6000: every LP targets SOC(24:00)=6000
REPORT_START = 31   # 0-based day index of 2025-02-01
STAGES = [6, 12, 18]
 
# ---------------------------------------------------------------- read data
def load_data():
    wbp = load_workbook(os.path.join(BASE, '附件1.xlsx'), read_only=True, data_only=True)
    rows = list(wbp[wbp.sheetnames[0]].iter_rows(values_only=True))[1:]
    price = np.array([float(r[1]) for r in rows])            # (144,) yuan/kWh
 
    wb2 = load_workbook(os.path.join(BASE, '附件2.xlsx'), read_only=True, data_only=True)
    rl = list(wb2['小区负载'].iter_rows(values_only=True))[1:]
    rp = list(wb2['光伏发电实际功率'].iter_rows(values_only=True))[1:]
    load = np.array([[float(v) for v in r[1:]] for r in rl])   # (365,144) kW
    pv = np.array([[float(v) for v in r[1:]] for r in rp])     # (365,144) kW
    dates = [r[0] for r in rl]
 
    wb3 = load_workbook(os.path.join(BASE, '附件3.xlsx'), read_only=True, data_only=True)
    r3 = list(wb3[wb3.sheetnames[0]].iter_rows(values_only=True))[1:]
    fc, cur = {}, None
    for row in r3:
        if row[0] is not None and str(row[0]).strip():
            cur = str(row[0]).strip()
            fc[cur] = {}
        fc[cur][str(row[1]).strip()] = np.array([float(v) for v in row[2:26]])
    day_of = {}
    for i, d in enumerate(dates):
        day_of[(d.month, d.day)] = i
    fc_by_day = {}
    for k, v in fc.items():
        y, m, d = (int(x) for x in k.split('-'))
        if (m, d) in day_of:
            fc_by_day[day_of[(m, d)]] = v
    return price, load, pv, fc_by_day, dates
 
PRICE, LOAD, PV, FC, DATES = load_data()
ELL = LOAD * DT      # load energy (365,144) kWh
ACT = PV * DT        # actual PV energy (365,144) kWh
 
# ------------------------------------------- forecast -> 10-min energy series
def forecast_profile(day, issue_hour, use_actual=False):
    """Return the (144,) planned PV energy series in kWh per slot.
 
    Hourly forecast values are linearly interpolated; the anchor at the
    issue time is the measured PV power of that same day.
    """
    if use_actual:
        return ACT[day].copy()
    H = np.zeros(25)
    H[0] = 0.0 if issue_hour == 0 else PV[day, issue_hour * 6 - 1]
    f = FC[day][f'{issue_hour}:00']
    for j in range(issue_hour + 1, 25):
        H[j] = f[j - issue_hour - 1]
    prof = np.zeros(N_T)
    for i in range(issue_hour * 6 + 1, N_T + 1):
        t = i * 10.0
        h = int(t // 60)
        r = (t - h * 60.0) / 60.0
        h0 = min(h, 23)
        if h >= 24:
            prof[i - 1] = H[24] * DT
        else:
            prof[i - 1] = (H[h0] * (1 - r) + H[h0 + 1] * r) * DT
    return prof
 
# ------------------------------------------------------------- LP solvers
def solve_lp(m, S_start, f, ell, price, x_orig=None):
    """LP over slots m..144. Returns dict(y, ch, dis, w, SOC).
 
    With x_orig given this is the adjustment LP: the settlement cost
    sum(p*(y + 0.5*|x-y|)) is linearized with the dp/dm variables.
    """
    n = N_T - m + 1
    idx = np.arange(m - 1, N_T)
    has_adj = x_orig is not None
    NV = 5 * n + (2 * n if has_adj else 0)
    iy, ic, idd, iw, iS = 0, n, 2 * n, 3 * n, 4 * n
    idp, idm = 5 * n, 6 * n
 
    c = np.zeros(NV)
    c[iy:iy + n] = price[idx]
    if has_adj:
        c[idp:idp + n] = K_PEN * price[idx]
        c[idm:idm + n] = K_PEN * price[idx]
 
    rows, cols, vals, beq = [], [], [], []
    r = 0
    # balance: y + dis - ch - w = ell - f
    for k in range(n):
        rows += [r, r, r, r]; cols += [iy + k, idd + k, ic + k, iw + k]
        vals += [1.0, 1.0, -1.0, -1.0]
        beq.append(ell[idx[k]] - f[idx[k]])
        r += 1
    # SOC recursion: S_k - S_{k-1} - ETA*ch + dis/ETA = 0 (first: = S_start)
    for k in range(n):
        rows += [r, r, r]; cols += [iS + k, ic + k, idd + k]
        vals += [1.0, -ETA, 1.0 / ETA]
        if k == 0:
            beq.append(S_start)
        else:
            rows += [r]; cols += [iS + k - 1]; vals += [-1.0]
            beq.append(0.0)
        r += 1
    # terminal constraint: S_144 = SOC_TGT
    rows += [r]; cols += [iS + n - 1]; vals += [1.0]; beq.append(SOC_TGT); r += 1
    A_eq = coo_matrix((vals, (rows, cols)), shape=(r, NV)).tocsr()
    b_eq = np.array(beq)
 
    Aub, bub = None, None
    if has_adj:
        rows, cols, vals, bub = [], [], [], []
        r = 0
        for k in range(n):
            # y - dp <= x_orig        (dp >= y - x)
            rows += [r, r]; cols += [iy + k, idp + k]; vals += [1.0, -1.0]
            bub.append(x_orig[idx[k]]); r += 1
            # -y - dm <= -x_orig      (dm >= x - y)
            rows += [r, r]; cols += [iy + k, idm + k]; vals += [-1.0, -1.0]
            bub.append(-x_orig[idx[k]]); r += 1
        Aub = coo_matrix((vals, (rows, cols)), shape=(r, NV)).tocsr()
        bub = np.array(bub)
 
    bounds = ([(0.0, None)] * n + [(0.0, E_MAX)] * n + [(0.0, E_MAX)] * n +
              [(0.0, None)] * n + [(S_MIN, S_MAX)] * n)
    if has_adj:
        bounds += [(0.0, None)] * (2 * n)
 
    res = linprog(c, A_ub=Aub, b_ub=bub, A_eq=A_eq, b_eq=b_eq,
                  bounds=bounds, method='highs')
    if not res.success:
        raise RuntimeError(f'LP failed m={m}: {res.message}')
    z = res.x
    return {'y': z[iy:iy + n], 'ch': z[ic:ic + n], 'dis': z[idd:idd + n],
            'w': z[iw:iw + n], 'S': z[iS:iS + n]}
 
# --------------------------------------------------------- real-time executor
def execute_interval(y, A, L, ch_p, dis_p, S):
    """Execute one slot with storage feedback. Returns (ch, dis, e, w, S_new)."""
    ch = min(ch_p, (S_MAX - S) / ETA, E_MAX)
    dis = min(dis_p, ETA * (S + ETA * ch - S_MIN), E_MAX)
    net = y + A + dis - L - ch
    if net < -1e-9:                       # deficit
        g = -net
        red = min(ch, g); ch -= red; g -= red
        if g > 1e-9:
            room = max(min(E_MAX - dis, ETA * (S + ETA * ch - S_MIN) - dis), 0.0)
            add = min(g, room); dis += add; g -= add
        e = max(g, 0.0); w = 0.0
    elif net > 1e-9:                      # surplus
        s = net
        red = min(dis, s); dis -= red; s -= red
        if s > 1e-9:
            room = max(min(E_MAX - ch, (S_MAX - S) / ETA - ch), 0.0)
            add = min(s, room); ch += add; s -= add
        w = max(s, 0.0); e = 0.0
    else:
        e = w = 0.0
    S_new = S + ETA * ch - dis / ETA
    S_new = min(max(S_new, S_MIN), S_MAX)
    return ch, dis, e, w, S_new
 
# ------------------------------------------------------------- day pipeline
def simulate_day(day, S_init):
    """Plan at 0:00, adjust at 6/12/18, execute slot by slot."""
    S = S_init
    f0 = forecast_profile(day, 0)
    lp0 = solve_lp(1, S, f0, ELL[day], PRICE)
    x_full = lp0['y'].copy()
    op_y = lp0['y'].copy(); op_ch = lp0['ch'].copy(); op_dis = lp0['dis'].copy()
 
    y_a = np.zeros(N_T); ch_a = np.zeros(N_T); dis_a = np.zeros(N_T)
    e_a = np.zeros(N_T); w_a = np.zeros(N_T)
    S_a = np.zeros(N_T + 1); S_a[0] = S
 
    bounds_list = [1] + [s * 6 + 1 for s in STAGES] + [N_T + 1]
    for si, m in enumerate(bounds_list[:-1]):
        m_end = bounds_list[si + 1]
        if si > 0:                        # adjustment LP issued at STAGES[si-1]
            h_issue = STAGES[si - 1]
            f_new = forecast_profile(day, h_issue)
            lp = solve_lp(m, S, f_new, ELL[day], PRICE, x_orig=x_full)
            op_y[m - 1:] = lp['y']; op_ch[m - 1:] = lp['ch']; op_dis[m - 1:] = lp['dis']
        for i in range(m, m_end):
            k = i - 1
            ch, dis, e, w, S = execute_interval(
                op_y[k], ACT[day, k], ELL[day, k], op_ch[k], op_dis[k], S)
            y_a[k] = op_y[k]; ch_a[k] = ch; dis_a[k] = dis
            e_a[k] = e; w_a[k] = w; S_a[k + 1] = S
 
    d_up = np.maximum(y_a - x_full, 0.0)
    d_dn = np.maximum(x_full - y_a, 0.0)
    settle = float((PRICE * (y_a + K_PEN * d_dn + K_PEN * d_up)).sum())
    plan_cost = float((PRICE * x_full).sum())
    emg_cost = float((K_EMG * PRICE * e_a).sum())
    # Store plain lists so the JSON export stays directly serializable.
    return {'x': x_full.tolist(), 'y': y_a.tolist(), 'ch': ch_a.tolist(),
            'dis': dis_a.tolist(), 'e': e_a.tolist(), 'w': w_a.tolist(),
            'S': S_a.tolist(), 'settle': settle, 'plan_cost': plan_cost,
            'emg_cost': emg_cost, 'total': settle + emg_cost}
 
# ----------------------------------------------------------------- run year
t0 = time.time()
results_all = []
s_init = S0
for day in range(LOAD.shape[0]):
    rec = simulate_day(day, s_init)
    rec['date'] = DATES[day].date().isoformat()
    results_all.append(rec)
    s_init = rec['S'][-1]
runtime = time.time() - t0
 
results = results_all[REPORT_START:]
 
# ------------------------------------------------------- summary statistics
emg_days = sum(bool((np.asarray(r['e']) > 1e-9).any()) for r in results)
emg_kwh = sum(float(np.sum(r['e'])) for r in results)
plan_cost_tot = sum(r['plan_cost'] for r in results)
settle_tot = sum(r['settle'] for r in results)
emg_cost_tot = sum(r['emg_cost'] for r in results)
curtail_tot = sum(float(np.sum(r['w'])) for r in results)
 
validation = {
    'date_count': len(results), 'expected_date_count': 334,
    'plan_slots_total': len(results) * N_T,
    'expected_plan_slots_total': 334 * N_T,
    'warmup_days': REPORT_START, 'warmup_initial_soc': S0,
    'soc_boundary': '连续跨日：S(day+1, 0) = S(day, 144)',
    'reported_start_soc': results[0]['S'][0],
    'reported_end_soc': results[-1]['S'][-1],
    'all_energy_unit': 'kWh', 'dt_hours': DT,
    'eta_c': ETA, 'eta_d': ETA, 'Smin': S_MIN, 'Smax': S_MAX,
    'Pmax_kW': P_MAX, 'Emax_kWh_per_10min': E_MAX,
    'terminal_rule': '每个LP固定SOC(24:00)=6000（fix6000）',
    'settlement_rule': 'min(x,y)按正常电价；(x-y)+加收50%；(y-x)+按150%',
    'emergency_rule': '缺口按5倍当时电价紧急购电；过剩弃光',
    'adjustment_stages': '06:00/12:00/18:00三次调整LP（B61218）',
    'forecast_rule': '附件3整点预报线性插值到10分钟网格，锚点取发布时刻实测光伏',
    'scenario_sources': '决策仅用当日已发布预报与历史储能状态，不用当日实测',
    'max_charge_kWh': max(float(np.max(r['ch'])) for r in results),
    'max_discharge_kWh': max(float(np.max(r['dis'])) for r in results),
    'soc_min_kWh': min(float(np.min(r['S'][1:])) for r in results),
    'soc_max_kWh': max(float(np.max(r['S'][1:])) for r in results),
    'emergency_days': emg_days,
    'emergency_energy_kWh': emg_kwh,
    'curtailment_kWh': curtail_tot,
    'plan_cost_total': plan_cost_tot,
    'settle_cost_total': settle_tot,
    'emergency_cost_total': emg_cost_tot,
    'total_cost_total': settle_tot + emg_cost_tot,
    'runtime_seconds': runtime,
}
 
# ------------------------------------------- independent audit (post-hoc)
audit = {'state': 0.0, 'actual_balance': 0.0, 'cost': 0.0,
         'cross_day': 0.0, 'simultaneous': 0.0}
previous_end = S0
for di, rec in enumerate(results_all):
    y, ch, dis = (np.asarray(rec[k]) for k in ['y', 'ch', 'dis'])
    e, w, state = (np.asarray(rec[k]) for k in ['e', 'w', 'S'])
    x = np.asarray(rec['x'])
    boundaries = state                      # length N_T + 1
    residual = np.diff(boundaries) - ETA * ch + dis / ETA
    balance = y + ACT[di] + dis + e - ELL[di] - ch - w
    settle_check = float(PRICE @ (y + K_PEN * np.maximum(x - y, 0.0)
                                  + K_PEN * np.maximum(y - x, 0.0)))
    emg_check = float((K_EMG * PRICE) @ e)
    audit['state'] = max(audit['state'], float(np.abs(residual).max()))
    audit['actual_balance'] = max(audit['actual_balance'],
                                  float(np.abs(balance).max()))
    audit['cost'] = max(audit['cost'], abs(settle_check + emg_check - rec['total']))
    audit['cross_day'] = max(audit['cross_day'],
                             abs(rec['S'][0] - previous_end))
    audit['simultaneous'] = max(audit['simultaneous'],
                                float(np.minimum(ch, dis).max()))
    assert min(boundaries) >= S_MIN - 1e-6
    assert max(boundaries) <= S_MAX + 1e-6
    assert min(ch) >= -1e-6 and min(dis) >= -1e-6
    assert max(ch) <= E_MAX + 1e-6 and max(dis) <= E_MAX + 1e-6
    previous_end = rec['S'][-1]
 
assert all(v < 1e-6 for v in audit.values())
 
with open(OUT, 'w', encoding='utf-8') as f:
    json.dump({'price': PRICE.tolist(),
               'results': results,
               'all_daily': [{'date': r['date'], 'soc_start': r['S'][0],
                              'soc_end': r['S'][-1], 'total': r['total']}
                             for r in results_all],
               'validation': validation}, f, ensure_ascii=False)
print(json.dumps(validation, ensure_ascii=False, indent=2))
 
# ------------------------------------------------------------- CSV exports
def tm(k):
    if k == 144:
        return '0:00+1'
    return f'{(k * 10) // 60}:{(k * 10) % 60:02d}'
 
def interval_label(start, end):
    return tm(start) + '-' + tm(end)
 
def write_csv(name, header, rows):
    with (args.output_dir / name).open('w', encoding='utf-8-sig', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(header)
        writer.writerows(rows)
 
write_csv('q3_10min.csv',
          ['date', 'period', 'x', 'y', 'c', 'r', 'soc_before', 'soc_after',
           'load_actual', 'pv_actual', 'emergency', 'unused', 'price',
           'emergency_cost'],
          ([rec['date'], interval_label(t, t + 1),
            rec['x'][t], rec['y'][t], rec['ch'][t], rec['dis'][t],
            rec['S'][t], rec['S'][t + 1],
            ELL[REPORT_START + ri][t], ACT[REPORT_START + ri][t],
            rec['e'][t], rec['w'][t], PRICE[t],
            5 * PRICE[t] * rec['e'][t]]
           for ri, rec in enumerate(results) for t in range(N_T)))
 
write_csv('q3_daily.csv',
          ['date', 'soc_start', 'soc_end', 'plan_kWh', 'actual_kWh',
           'emergency_kWh', 'plan_cost', 'settle_cost', 'emergency_cost',
           'total_cost'],
          ([rec['date'], rec['S'][0], rec['S'][-1], sum(rec['x']),
            sum(rec['y']), sum(rec['e']), rec['plan_cost'], rec['settle'],
            rec['emg_cost'], rec['total']] for rec in results))
 
def urgent_intervals(e):
    out, t = [], 0
    while t < N_T:
        if e[t] <= 1e-9:
            t += 1
            continue
        st, amount = t, 0.0
        while t < N_T and e[t] > 1e-9:
            amount += e[t]
            t += 1
        out.append({'period': interval_label(st, t), 'amount': float(amount),
                    'start': st, 'end': t})
    return out
 
for r in results:
    r['urgent_intervals'] = urgent_intervals(np.asarray(r['e']))
 
write_csv('q3_emergency.csv', ['date', 'period', 'kWh'],
          ([rec['date'], q['period'], q['amount']]
           for rec in results for q in rec['urgent_intervals']))
 
# ---------------------------------------------------- optional Excel check
# The official answer template rotates columns by one data-row label:
# column 'a-b' holds the array slot ending at a, i.e. array index j-1.
# Read only: the supplied answer workbook is never changed.
if args.answer is not None:
    answer = load_workbook(args.answer, read_only=True, data_only=True)
    plan_error = adjust_error = storage_error = emergency_error = 0.0
 
    # Sheet 1: 0:00 planned purchase x; columns 2..145 hold array x[0..143]
    # under the one-label rotation, 146 = daily total, 147 = plan cost.
    rows = list(answer['计划购电量'].iter_rows(values_only=True))[1:]
    assert len(rows) == len(results)
    for ri, rec in enumerate(results):
        row = rows[ri]
        assert row[0].date().isoformat() == rec['date']
        for j in range(1, N_T + 1):
            plan_error = max(plan_error, abs(float(row[j]) - rec['x'][j - 1]))
        assert abs(float(row[145]) - sum(rec['x'])) < 1e-3
        assert abs(float(row[146]) - rec['plan_cost']) < 1e-2
 
    # Sheet 2: executed purchase y; 147 = settle + emergency total cost.
    rows = list(answer['调整购电量'].iter_rows(values_only=True))[1:]
    for ri, rec in enumerate(results):
        row = rows[ri]
        for j in range(1, N_T + 1):
            adjust_error = max(adjust_error,
                               abs(float(row[j]) - rec['y'][j - 1]))
        assert abs(float(row[145]) - sum(rec['y'])) < 1e-3
        assert abs(float(row[146]) - rec['total']) < 1e-2
 
    # Sheet 3: six 4-hour blocks per day; SOC in the first two rows.
    rows3 = list(answer['充放电量'].iter_rows(values_only=True))[1:]
    assert len(rows3) == len(results) * 6
    for ri, rec in enumerate(results):
        block = rows3[ri * 6:(ri + 1) * 6]
        for b in range(6):
            expected_c = float(sum(rec['ch'][b * 24:(b + 1) * 24]))
            expected_d = float(sum(rec['dis'][b * 24:(b + 1) * 24]))
            storage_error = max(storage_error,
                                abs(float(block[b][2]) - expected_c),
                                abs(float(block[b][3]) - expected_d))
        assert abs(float(block[0][5]) - rec['S'][0]) < 1e-3
        assert abs(float(block[1][5]) - rec['S'][-1]) < 1e-3
 
    # Sheet 4: one row per positive emergency run; the date cell is
    # written only on the first row of each day's runs (merged style).
    rows4 = list(answer['紧急购电量'].iter_rows(values_only=True))[1:]
    expected = [(rec['date'], q['period'], q['amount'])
                for rec in results for q in rec['urgent_intervals']]
    assert len(rows4) == len(expected)
    cur_date = None
    for row, item in zip(rows4, expected):
        if row[0] is not None:
            cur_date = row[0].date().isoformat()
        assert cur_date == item[0]
        assert row[1] == item[1]
        emergency_error = max(emergency_error, abs(float(row[2]) - item[2]))
    answer.close()
    assert max(plan_error, adjust_error, storage_error,
               emergency_error) < 1e-3     # template stores 4 decimals
    audit.update(workbook_plan=plan_error, workbook_adjust=adjust_error,
                 workbook_storage=storage_error, workbook_emergency=emergency_error)
 
with (args.output_dir / 'q3_checks.json').open('w', encoding='utf-8') as f:
    json.dump(audit, f, ensure_ascii=False, indent=2)
 
print('Independent checks:', json.dumps(audit, ensure_ascii=False))


In [ ]:
"""Q4 recomputation. Q4-2 inherits the verified Q2 scenario LP.
Q4-3 implements the supplied Problem3 rolling LP and causal feedback.
No original result workbook is edited or used as optimization input.
"""
import sys,json,time,hashlib
from pathlib import Path
import numpy as np
from scipy.optimize import linprog
from scipy.sparse import lil_matrix
from openpyxl import load_workbook
 
import argparse, csv
parser=argparse.ArgumentParser(description='Reproduce the Q4 model and all comparison experiments.')
parser.add_argument('--data-dir',type=Path,default=Path('附件'))
parser.add_argument('--output-dir',type=Path,default=Path('recomputed_results'))
args=parser.parse_args()
BASE=args.data_dir
WORK=args.output_dir;WORK.mkdir(parents=True,exist_ok=True)
N=144;ETA=.9;EMAX=5000/6;SMIN=1200.;SMAX=10800.
def read(name):
 w=load_workbook(BASE/name,read_only=True,data_only=True);a={s.title:list(s.values) for s in w};w.close();return a
a1=read('附件1.xlsx')['Sheet1'];a2=read('附件2.xlsx');a3=read('附件3.xlsx')['Sheet1'];a4=read('附件4.xlsx')['Sheet1']
dates=[r[0].date().isoformat() for r in a2['小区负载'][1:]]
L=np.array([r[1:] for r in a2['小区负载'][1:]],float)/6
G=np.array([r[1:] for r in a2['光伏发电实际功率'][1:]],float)/6
P=np.array([r[1:] for r in a4[1:]],float)
PF=np.tile(np.array([r[1] for r in a1[1:]],float),(365,1))
BL=np.array([r[2] for r in a1[1:]],float)/6;BG=np.array([r[3] for r in a1[1:]],float)/6
F=np.array([r[2:] for r in a3[1:]],float).reshape(365,4,24)
assert L.shape==G.shape==P.shape==(365,144)
assert np.all(P>0) and np.all(F>=0)
assert [r[0].date().isoformat() for r in a4[1:]]==dates
 
def clean(z):
 z=np.asarray(z).copy();z[np.abs(z)<1e-9]=0;return z
def solve(obj,A,b,bounds):
 res=linprog(obj,A_eq=A,b_eq=b,bounds=bounds,method='highs',options={'dual_feasibility_tolerance':1e-8,'primal_feasibility_tolerance':1e-8})
 if not res.success: raise RuntimeError(res.message)
 return clean(res.x),float(res.fun)
def label(t):return '24:00' if t==144 else f'{t//6:02d}:{t%6*10:02d}'
def intervals(e):
 out=[];t=0
 while t<144:
  if e[t]<=1e-7:t+=1;continue
  st=t
  while t<144 and e[t]>1e-7:t+=1
  out.append({'start':st,'end':t,'period':label(st)+'-'+label(t),'energy':float(sum(e[st:t]))})
 return out
def record(di,p,x,y,c,r,S,e,w,updates=None,forecast=None):
 up=np.maximum(y-x,0);down=np.maximum(x-y,0)
 # Settlement retained from supplied Problem3: retained plan + cancellation penalty + increased volume.
 retained=np.minimum(x,y);pc=float(p@x);uc=float(1.5*p@up);dc=float(.5*p@down);refund=float(p@down)
 ec=float(5*p@e);normal=pc-refund+uc+dc
 rec=dict(date=dates[di],price=p.tolist(),load=L[di].tolist(),pv=G[di].tolist(),x=x.tolist(),y=y.tolist(),
  c=c.tolist(),r=r.tolist(),S=S.tolist(),e=e.tolist(),w=w.tolist(),up=up.tolist(),down=down.tolist(),
  plan_cost=pc,retained_plan_cost=float(p@retained),cancellation_refund=refund,increase_cost=uc,cancellation_penalty=dc,
  adjustment_net_cost=uc+dc-refund,normal_cost=normal,emergency_cost=ec,total_cost=normal+ec,
  intervals=intervals(e),updates=updates or [],forecast=forecast)
 bal=y+G[di]+r+e-L[di]-c-w;state=np.diff(S)-ETA*c+r/ETA
 rec['checks']=dict(balance=float(abs(bal).max()),state=float(abs(state).max()),Smin=float(min(S)),Smax=float(max(S)),
   Cmax=float(max(c)),Rmax=float(max(r)),simultaneous=float(np.minimum(c,r).max()),last_state=float(abs(state[-1])),
   cost=float(abs(normal-(p@(y+.5*abs(y-x))))))
 assert max(abs(bal).max(),abs(state).max())<1e-6
 assert min(S)>SMIN-1e-6 and max(S)<SMAX+1e-6
 assert max(max(c),max(r))<EMAX+1e-6 and min(min(c),min(r),min(e),min(w))>-1e-7
 assert np.minimum(c,r).max()<1e-6,'simultaneous operation requires review'
 return rec
 
# Q2 common controls x,c,r,S, predicted unused, 7 scenario emergency, 7 scenario unused.
nv=19*N;neq=9*N;A=lil_matrix((neq,nv));row=0
for t in range(N):
 A[row,t]=1;A[row,2*N+t]=1;A[row,N+t]=-1;A[row,4*N+t]=-1;row+=1
for s in range(7):
 for t in range(N):
  A[row,t]=1;A[row,2*N+t]=1;A[row,5*N+s*N+t]=1;A[row,N+t]=-1;A[row,12*N+s*N+t]=-1;row+=1
for t in range(N):
 A[row,3*N+t]=1;A[row,N+t]=-ETA;A[row,2*N+t]=1/ETA
 if t:A[row,3*N+t-1]=-1
 row+=1
A=A.tocsr()
def q2_day(di,s0,p):
 if di>=7:hl,hg=L[di-7:di],G[di-7:di]
 else:hl=np.vstack([np.tile(BL,(7-di,1)),L[:di]]);hg=np.vstack([np.tile(BG,(7-di,1)),G[:di]])
 lm,gm=hl.mean(0),hg.mean(0)
 b=np.r_[lm-gm,(hl-hg).ravel(),s0,np.zeros(N-1)]
 obj=np.r_[p,np.zeros(4*N),np.tile(5*p/7,7),np.zeros(7*N)]
 bounds=[(0,None)]*nv
 bounds[N:3*N]=[(0,EMAX)]*(2*N);bounds[3*N:4*N]=[(SMIN,SMAX)]*N
 bounds[4*N:5*N]=[(0,v) for v in gm]
 z,fun=solve(obj,A,b,bounds)
 x,c,r=z[:N],z[N:2*N],z[2*N:3*N];S=np.r_[s0,z[3*N:4*N]]
 e=clean(np.maximum(L[di]+c-x-G[di]-r,0));w=clean(np.maximum(x+G[di]+r-L[di]-c,0))
 rec=record(di,p,x,x.copy(),c,r,S,e,w,forecast={'load':lm.tolist(),'pv':gm.tolist(),'history_end':dates[di-1] if di else None})
 rec['objective']=fun;rec['checks']['lp']=float(np.abs(A@z-b).max())
 return rec
 
def forecast_at(di,k):
 start=36*k
 anchor=0. if k==0 else G[di,start-1]*6
 # Forecast h refers to the point release+h, not the preceding whole hour.
 return np.interp(np.arange(start+1,145)/6,np.arange(25)+k*6,np.r_[anchor,F[di,k]])/6
 
LP_CACHE={}
def q3_lp(l,f,p,s0,x=None):
 n=len(l);adjust=x is not None;nv=(7 if adjust else 5)*n
 key=(n,adjust)
 if key not in LP_CACHE:
  a=lil_matrix((2*n+1+(n if adjust else 0),nv));row=0
  for t in range(n):
   a[row,t]=1;a[row,2*n+t]=1;a[row,n+t]=-1;a[row,4*n+t]=-1;row+=1
  for t in range(n):
   a[row,3*n+t]=1;a[row,n+t]=-ETA;a[row,2*n+t]=1/ETA
   if t:a[row,3*n+t-1]=-1
   row+=1
  a[row,4*n-1]=1;row+=1
  if adjust:
   for t in range(n):a[row,t]=1;a[row,5*n+t]=-1;a[row,6*n+t]=1;row+=1
  LP_CACHE[key]=a.tocsr()
 a=LP_CACHE[key]
 b=np.r_[l-f,s0,np.zeros(n-1),6000,x if adjust else []]
 obj=np.zeros(nv);obj[:n]=p
 if adjust:obj[5*n:6*n]=.5*p;obj[6*n:7*n]=.5*p
 bounds=[(0,None)]*nv;bounds[n:3*n]=[(0,EMAX)]*(2*n);bounds[3*n:4*n]=[(SMIN,SMAX)]*n
 bounds[4*n:5*n]=[(0,v) for v in f]
 z,fun=solve(obj,a,b,bounds)
 assert np.abs(a@z-b).max()<1e-6
 # Select a non-cycling solution among cost-optimal degenerate LP solutions.
 # The primary cost is retained within 1e-7 yuan; this is not a new weighted cost objective.
 if np.minimum(z[n:2*n],z[2*n:3*n]).max()>1e-6:
  tie=np.zeros(nv);tie[n:3*n]=1
  res=linprog(tie,A_eq=a,b_eq=b,A_ub=obj.reshape(1,-1),b_ub=[fun+1e-7],bounds=bounds,method='highs',options={'dual_feasibility_tolerance':1e-8,'primal_feasibility_tolerance':1e-8})
  if not res.success:raise RuntimeError(res.message)
  z=clean(res.x)
  assert obj@z<=fun+1e-6
 assert np.abs(a@z-b).max()<1e-6
 assert np.minimum(z[n:2*n],z[2*n:3*n]).max()<1e-6
 return z[:n],z[n:2*n],z[2*n:3*n],fun
 
def q3_day(di,s0,p,mask=(1,2,3)):
 f0=forecast_at(di,0);x,cp,rp,fun=q3_lp(L[di],f0,p,s0)
 y=x.copy();c=np.zeros(N);r=np.zeros(N);e=np.zeros(N);w=np.zeros(N);S=np.r_[s0,np.zeros(N)]
 updates=[dict(release='00:00',start=0,purchase=x.tolist(),charge=cp.tolist(),discharge=rp.tolist(),forecast=f0.tolist(),state=float(s0))]
 for t in range(N):
  if t in (36,72,108) and t//36 in mask:
   k=t//36;f=forecast_at(di,k);yp,cpp,rpp,_=q3_lp(L[di,t:],f,p[t:],S[t],x[t:])
   y[t:]=yp;cp[t:]=cpp;rp[t:]=rpp
   updates.append(dict(release=label(t),start=t,purchase=yp.tolist(),charge=cpp.tolist(),discharge=rpp.tolist(),forecast=f.tolist(),state=float(S[t])))
  ct=min(cp[t],EMAX,max(0,(SMAX-S[t])/ETA));rt=min(rp[t],EMAX,max(0,(S[t]-SMIN)*ETA))
  net=y[t]+G[di,t]+rt-L[di,t]-ct
  if net<0:
   delta=min(ct,-net);ct-=delta;net+=delta
   delta=min(-net,EMAX-rt,max(0,(S[t]+ETA*ct-SMIN)*ETA-rt));rt+=delta;net+=delta
   e[t]=max(0,-net)
  else:
   delta=min(rt,net);rt-=delta;net-=delta
   delta=min(net,EMAX-ct,max(0,(SMAX-S[t]+rt/ETA)/ETA-ct));ct+=delta;net-=delta
   w[t]=max(0,net)
  c[t]=ct;r[t]=rt;S[t+1]=S[t]+ETA*ct-rt/ETA
 return record(di,p,x,y,clean(c),clean(r),S,clean(e),clean(w),updates,{'load':L[di].tolist(),'pv':f0.tolist()})
 
def summary(rs):
 fields=['plan_cost','retained_plan_cost','cancellation_refund','increase_cost','cancellation_penalty','adjustment_net_cost','normal_cost','emergency_cost','total_cost']
 out={k:sum(r[k] for r in rs) for k in fields}
 out.update({k+'_energy':sum(sum(r[k]) for r in rs) for k in ['x','y','c','r','e','w','up','down']})
 out.update(days=len(rs),emergency_days=sum(bool(r['intervals']) for r in rs),emergency_intervals=sum(len(r['intervals']) for r in rs),
  start_soc=rs[0]['S'][0],end_soc=rs[-1]['S'][-1],min_soc=min(r['checks']['Smin'] for r in rs),max_soc=max(r['checks']['Smax'] for r in rs),
  max_charge=max(r['checks']['Cmax'] for r in rs),max_discharge=max(r['checks']['Rmax'] for r in rs),
  max_balance=max(r['checks']['balance'] for r in rs),max_state=max(r['checks']['state'] for r in rs),
  max_last_state=max(r['checks']['last_state'] for r in rs),max_simultaneous=max(r['checks']['simultaneous'] for r in rs))
 return out
 
def main():
 t0=time.time();summaries={}
 jobs=[('q42_variable','q2',P,()),('q42_fixed','q2',PF,()),('q43_variable','q3',P,(1,2,3)),('q43_fixed','q3',PF,(1,2,3)),
       ('q43_initial','q3',P,()),('q43_06','q3',P,(1,)),('q43_0612','q3',P,(1,2))]
 for name,method,pp,mask in jobs:
  dest=WORK/(name+'.json')
  records=[];s0=6000.
  for di in range(365):
   rec=q2_day(di,s0,pp[di]) if method=='q2' else q3_day(di,s0,pp[di],mask)
   assert abs(rec['S'][0]-s0)<1e-8
   s0=rec['S'][-1]
   records.append(rec)
   if di%90==0:print(name,di,flush=True)
  sums=summary(records[31:]);summaries[name]=sums
  dest.write_text(json.dumps({'summary':sums,'results':records[31:],'warmup':records[:31]},ensure_ascii=False))
  print(name,json.dumps(sums),flush=True)
 (WORK/'summary.json').write_text(json.dumps(summaries,ensure_ascii=False,indent=2))
 
 for name in ['q42_variable','q43_variable']:
  result=json.loads((WORK/(name+'.json')).read_text())['results']
  with (WORK/(name+'_10min.csv')).open('w',encoding='utf-8-sig',newline='') as f:
   wr=csv.writer(f)
   wr.writerow(['date','period','price','load','pv','plan','final_purchase','charge','discharge','soc_before','soc_after','emergency','unused','normal_cost','emergency_cost'])
   for rec in result:
    for t in range(144):
     p,x,y,e=[rec[k][t] for k in ['price','x','y','e']]
     wr.writerow([rec['date'],label(t)+'-'+label(t+1),p,rec['load'][t],rec['pv'][t],x,y,rec['c'][t],rec['r'][t],rec['S'][t],rec['S'][t+1],e,rec['w'][t],p*(y+.5*abs(y-x)),5*p*e])
 sources={f.name:hashlib.sha256(f.read_bytes()).hexdigest() for f in BASE.glob('附件*.xlsx')}
 (WORK/'provenance.json').write_text(json.dumps({'source_sha256':sources,'seconds':time.time()-t0,'solver':'scipy.optimize.linprog / HiGHS','scenario_window':7,'interval_hours':'1/6','energy_cap':'5000/6','q3_settlement':'p*(y+0.5*abs(y-x))','price_information':'Full same-day attachment4 curve treated as supplied known input; no price forecasting or clairvoyance optimality claim.'},ensure_ascii=False,indent=2))
 print('FINISHED seconds',time.time()-t0,flush=True)
if __name__=='__main__':main()
Excel结果生成程序
以下为生成result4-2.xlsx和result4-3.xlsx的完整Node.js交互脚本。它读取文件列表中的两个workbook JSON，不重新求解模型；Excel中的费用公式、显示精度、冻结窗格和工作表名称均由脚本写入。
import fs from 'node:fs/promises';
import {Workbook,SpreadsheetFile} from '@oai/artifact-tool';
const dir=new URL('.',import.meta.url).pathname;
const out=new URL('../../outputs/q4_final_01a090f0/',import.meta.url).pathname;
// Resolve output relative to the project root (builder resides in work/q4_final).
const outputDir=dir+'../../outputs/q4_final_01a090f0/';
await fs.mkdir(outputDir,{recursive:true});
const date=s=>new Date(s+'T00:00:00Z');
const time=t=>t===144?'24:00':`${String(Math.floor(t/6)).padStart(2,'0')}:${String(t%6*10).padStart(2,'0')}`;
const periods=Array.from({length:144},(_,t)=>time(t)+'-'+time(t+1));
function style(s,rows,cols,width=15){
 const used=s.getRangeByIndexes(0,0,rows,cols);
 used.format.font={name:'PingFang SC',size:10,color:'#202020'};used.format.rowHeight=22;used.format.verticalAlignment='center';
 used.format.columnWidth=width;s.showGridLines=false;
 const head=s.getRangeByIndexes(0,0,1,cols);head.format.fill='#E7EDF4';head.format.font.bold=true;head.format.rowHeight=40;head.format.wrapText=true;head.format.horizontalAlignment='center';
 s.getRangeByIndexes(1,1,rows-1,cols-1).setNumberFormat('#,##0.00');
 s.getRangeByIndexes(1,0,rows-1,1).setNumberFormat('yyyy-mm-dd');s.getRangeByIndexes(0,0,rows,1).format.columnWidth=15;s.getRangeByIndexes(1,0,rows-1,1).format.horizontalAlignment='center';
 s.freezePanes.freezeRows(1);s.freezePanes.freezeColumns(1);
}
for(const tag of ['4-2','4-3']){
 const d=JSON.parse(await fs.readFile(dir+`workbook_${tag}.json`,'utf8'));const wb=Workbook.create();
 const names=['计划购电量',...(tag==='4-3'?['调整购电量']:[]),'充放电量','紧急购电量','费用汇总','逐时紧急购电','附件4电价'];
 const sh=Object.fromEntries(names.map(n=>[n,wb.worksheets.add(n)]));
 for(const [name,key] of [['计划购电量','plan'],...(tag==='4-3'?[['调整购电量','adjusted']]:[])]){
  const s=sh[name];s.getRange('A1:EQ335').values=[['日期',...periods,'全天购电量','全天购电费'],...d.dates.map((x,i)=>[date(x),...d[key][i],null,null])];
  s.getRange('EP2').formulas=[['=SUM(B2:EO2)']];s.getRange('EP2:EP335').fillDown();
  s.getRange('EQ2').formulas=[[name==='计划购电量'?"=SUMPRODUCT(B2:EO2,'附件4电价'!B2:EO2)":"='费用汇总'!K2"]];s.getRange('EQ2:EQ335').fillDown();style(s,335,147,16);
 }
 for(const [name,key] of [['逐时紧急购电','emergency'],['附件4电价','price']]){
  const s=sh[name];s.getRange('A1:EO335').values=[['日期',...periods],...d.dates.map((x,i)=>[date(x),...d[key][i]])];style(s,335,145,16);
  if(key==='price')s.getRange('B2:EO335').setNumberFormat('0.0000');
 }
 sh['附件4电价'].getRange('A337').values=[['来源：附件4.xlsx；原采样标签为时段终点，本表显示实际10分钟区间。']];
 const s=sh['充放电量'];s.getRangeByIndexes(0,0,d.storage.length+1,6).values=[['日期','时间段','充电量（kWh）','放电量（kWh）','时刻','储电量（kWh）'],...d.storage.map(r=>[date(r[0]),...r.slice(1)])];style(s,d.storage.length+1,6,21);s.getRangeByIndexes(1,1,d.storage.length,1).setNumberFormat('@');s.getRangeByIndexes(1,4,d.storage.length,1).setNumberFormat('@');
 const em=sh['紧急购电量'];em.getRangeByIndexes(0,0,d.emergency_intervals.length+1,3).values=[['日期','紧急购电时间段','紧急购电量（kWh）'],...d.emergency_intervals.map(r=>[date(r[0]),r[1],r[2]])];style(em,d.emergency_intervals.length+1,3,25);em.getRangeByIndexes(1,1,d.emergency_intervals.length,1).setNumberFormat('@');
 const co=sh['费用汇总'];co.getRange('A1:K335').values=[['日期','计划购电量\n(kWh)','最终常规购电量\n(kWh)','紧急购电量\n(kWh)','原计划费用\n(元)','取消部分原价冲减\n(元)','增购费用\n(元)','取消违约费用\n(元)','紧急购电费用\n(元)','常规结算费用\n(元)','实际总费用\n(元)'],...d.costs.map(r=>[date(r[0]),...r.slice(1)])];
 co.getRange('B2:E2').formulas=[["='计划购电量'!EP2",tag==='4-3'?"='调整购电量'!EP2":"='计划购电量'!EP2","=SUM('逐时紧急购电'!B2:EO2)","='计划购电量'!EQ2"]];co.getRange('B2:E335').fillDown();
 co.getRange('I2:K2').formulas=[["=5*SUMPRODUCT('逐时紧急购电'!B2:EO2,'附件4电价'!B2:EO2)",'=E2-F2+G2+H2','=J2+I2']];co.getRange('I2:K335').fillDown();
 co.getRange('A336').values=[['报告期合计']];co.getRange('B336:K336').formulas=[Array.from({length:10},(_,i)=>`=SUM(${String.fromCharCode(66+i)}2:${String.fromCharCode(66+i)}335)`)];style(co,336,11,20);co.getRange('A336:K336').format.font.bold=true;co.getRange('A336:K336').format.fill='#E7EDF4';co.getRange('A336').setNumberFormat('@');
 co.getRange('A338').values=[['说明：计划表末列为计划费用，调整表末列为含紧急购电的实际总费用。']];
 co.getRange('A339').values=[['F至H列为逐时方案的汇总值；对应逐10分钟明细和完整精度均保存在复现附件中。']];
 s.getRangeByIndexes(1,1,d.storage.length,1).format.horizontalAlignment='center';em.getRangeByIndexes(1,1,d.emergency_intervals.length,1).format.horizontalAlignment='center';
 wb.recalculate();
 console.log(tag,(await wb.inspect({kind:'table',range:'费用汇总!A336:K336',include:'values,formulas',tableMaxRows:1,tableMaxCols:11,maxChars:1800})).ndjson);
 console.log(tag,(await wb.inspect({kind:'match',searchTerm:'#REF!|#DIV/0!|#VALUE!|#NAME\\?|#NUM!|#SPILL!',options:{useRegex:true,maxResults:10},maxChars:1200})).ndjson);
 for(const name of names){
  const range=name==='充放电量'?'A1:F8':name==='紧急购电量'?'A1:C8':name==='费用汇总'?'A1:F7':'A1:E7';
  const png=await wb.render({sheetName:name,range,scale:1.5,format:'png'});await fs.writeFile(dir+`preview_${tag}_${name}.png`,new Uint8Array(await png.arrayBuffer()));
 }
 const f=await SpreadsheetFile.exportXlsx(wb);await f.save(outputDir+`result${tag}.xlsx`);console.log('EXPORTED',tag);
}
